# Brain Tumor MRI Classification — PyTorch vs TensorFlow

Compare **ResNet-50** and **EfficientNet-B0** on the same brain tumor MRI dataset, with Grad-CAM explanations in each framework.

In [ ]:
import os
import json

# Kaggle path first; fall back to local clone of the dataset
KAGGLE_DATA = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
LOCAL_DATA = os.path.join("..", "data", "brain-tumor-mri-dataset")
DATA_DIR = KAGGLE_DATA if os.path.isdir(KAGGLE_DATA) else LOCAL_DATA

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Dataset not found at {KAGGLE_DATA!r} or {LOCAL_DATA!r}. "
        "Download the Brain Tumor MRI dataset and point DATA_DIR to it."
    )

TRAIN_DIR = os.path.join(DATA_DIR, "Training")
TEST_DIR = os.path.join(DATA_DIR, "Testing")
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
ARCHITECTURES = ["resnet50", "efficientnet_b0"]

# Set True to ignore saved checkpoints and retrain everything
RETRAIN = False

MODEL_DIR = os.path.join("..", "model")
PT_CKPT_DIR = os.path.join(MODEL_DIR, "pytorch")
TF_CKPT_DIR = os.path.join(MODEL_DIR, "tensorflow")
os.makedirs(PT_CKPT_DIR, exist_ok=True)
os.makedirs(TF_CKPT_DIR, exist_ok=True)


def _load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _save_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def pt_checkpoint_path(arch):
    return os.path.join(PT_CKPT_DIR, f"{arch}.pt")


def pt_history_path(arch):
    return os.path.join(PT_CKPT_DIR, f"{arch}_history.json")


def tf_checkpoint_path(arch):
    return os.path.join(TF_CKPT_DIR, f"{arch}.keras")


def tf_history_path(arch):
    return os.path.join(TF_CKPT_DIR, f"{arch}_history.json")


def tf_meta_path(arch):
    return os.path.join(TF_CKPT_DIR, f"{arch}_meta.json")

print("DATA_DIR:", DATA_DIR)
print("RETRAIN:", RETRAIN)
print("Checkpoints:", MODEL_DIR)


## PyTorch

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class TransformSubset(Dataset):
    """Apply a transform only to indices from a random_split subset."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = self.subset.dataset.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, label


base_train = datasets.ImageFolder(TRAIN_DIR)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
class_names = base_train.classes
print("Classes:", class_names)

val_size = int(0.15 * len(base_train))
train_size = len(base_train) - val_size
train_sub, val_sub = random_split(
    base_train, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_dataset = TransformSubset(train_sub, train_transform)
val_dataset = TransformSubset(val_sub, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
def build_pytorch_model(architecture: str, num_classes: int) -> nn.Module:
    architecture = architecture.lower()
    if architecture == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for param in model.parameters():
            param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        cam_layer_getter = lambda m: m.layer4[-1]
    elif architecture == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        for param in model.parameters():
            param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        cam_layer_getter = lambda m: m.features[-1]
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    return model.to(device), cam_layer_getter


def train_pytorch_model(model, train_loader, val_loader, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    head_params = (p for p in model.parameters() if p.requires_grad)
    optimizer = optim.Adam(head_params, lr=1e-3)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(
            f"Epoch {epoch + 1}/{epochs} — "
            f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f}, acc {val_acc:.4f}"
        )
    return history


def evaluate_pytorch_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    report = classification_report(all_labels, all_preds, target_names=class_names)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "report": report,
        "y_true": all_labels,
        "y_pred": all_preds,
    }

def pytorch_cam_layer(model, architecture: str):
    architecture = architecture.lower()
    if architecture == "resnet50":
        return model.layer4[-1]
    if architecture == "efficientnet_b0":
        return model.features[-1]
    raise ValueError(f"Unknown architecture: {architecture}")


In [ ]:
pt_models = {}
pt_histories = {}
pt_metrics = {}
pt_cam_layers = {}

for arch in ARCHITECTURES:
    print("\n" + "=" * 60)
    ckpt = pt_checkpoint_path(arch)
    hist_path = pt_history_path(arch)

    legacy_pt = os.path.join(MODEL_DIR, "model_T08.pt")
    if (
        not RETRAIN
        and not os.path.isfile(ckpt)
        and arch == "resnet50"
        and os.path.isfile(legacy_pt)
    ):
        print(f"Adopting legacy PyTorch export for {arch}: {legacy_pt}")
        model = torch.load(legacy_pt, map_location=device, weights_only=False)
        model = model.to(device)
        model.eval()
        torch.save(model, ckpt)
        print(f"Wrote per-arch checkpoint: {ckpt}")

    if not RETRAIN and os.path.isfile(ckpt):
        print(f"PyTorch — loading cached {arch}")
        print("=" * 60)
        model = torch.load(ckpt, map_location=device, weights_only=False)
        model = model.to(device)
        model.eval()
        if os.path.isfile(hist_path):
            pt_histories[arch] = _load_json(hist_path)
        else:
            pt_histories[arch] = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
            print("(No history JSON — training curves empty for this arch.)")
    else:
        print(f"PyTorch — training {arch}")
        print("=" * 60)
        model, _ = build_pytorch_model(arch, len(class_names))
        pt_histories[arch] = train_pytorch_model(model, train_loader, val_loader)
        torch.save(model, ckpt)
        _save_json(hist_path, pt_histories[arch])
        print(f"Saved checkpoint: {ckpt}")

    pt_models[arch] = model
    pt_metrics[arch] = evaluate_pytorch_model(model, test_loader)
    pt_cam_layers[arch] = pytorch_cam_layer(model, arch)

    print(f"\nTest accuracy ({arch}): {pt_metrics[arch]['accuracy']:.4f}")
    print(pt_metrics[arch]["report"])


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, arch in enumerate(ARCHITECTURES):
    h = pt_histories[arch]
    axes[i, 0].plot(h["train_loss"], label="Train")
    axes[i, 0].plot(h["val_loss"], label="Val")
    axes[i, 0].set_title(f"{arch} — loss")
    axes[i, 0].legend()

    axes[i, 1].plot(h["train_acc"], label="Train")
    axes[i, 1].plot(h["val_acc"], label="Val")
    axes[i, 1].set_title(f"{arch} — accuracy")
    axes[i, 1].legend()

plt.tight_layout()
plt.show()

pt_summary = [
    {
        "framework": "pytorch",
        "architecture": arch,
        "accuracy": pt_metrics[arch]["accuracy"],
        "precision": pt_metrics[arch]["precision"],
        "recall": pt_metrics[arch]["recall"],
        "f1": pt_metrics[arch]["f1"],
    }
    for arch in ARCHITECTURES
]
print("PyTorch comparison:")
for row in pt_summary:
    print(row)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


def denormalize_imagenet(tensor_chw):
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std = np.array(IMAGENET_STD, dtype=np.float32)
    img = tensor_chw.numpy().transpose(1, 2, 0)
    img = img * std + mean
    return np.clip(img, 0.0, 1.0)


def run_pytorch_gradcam(model, target_layer, sample_tensor, target_class=None):
    for param in target_layer.parameters():
        param.requires_grad = True

    cam = GradCAM(model=model, target_layers=[target_layer])
    if target_class is None:
        with torch.no_grad():
            target_class = model(sample_tensor.unsqueeze(0).to(device)).argmax(1).item()
    targets = [ClassifierOutputTarget(target_class)]
    try:
        grayscale_cam = cam(input_tensor=sample_tensor.unsqueeze(0).to(device), targets=targets)[0]
    finally:
        if hasattr(cam, "activations_and_grads"):
            cam.activations_and_grads.release()
    return grayscale_cam, target_class


sample_img, sample_label = test_dataset[0]
rgb_base = denormalize_imagenet(sample_img)

fig, axes = plt.subplots(len(ARCHITECTURES), 2, figsize=(10, 4 * len(ARCHITECTURES)))
if len(ARCHITECTURES) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, arch in enumerate(ARCHITECTURES):
    model = pt_models[arch]
    model.eval()
    heatmap, pred_class = run_pytorch_gradcam(
        model, pt_cam_layers[arch], sample_img, target_class=sample_label
    )
    overlay = show_cam_on_image(rgb_base.astype(np.float32), heatmap, use_rgb=True)

    axes[row, 0].imshow(rgb_base)
    axes[row, 0].set_title(f"{arch} — original ({class_names[sample_label]})")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title(f"Grad-CAM → {class_names[pred_class]}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
best_pt_arch = max(ARCHITECTURES, key=lambda a: pt_metrics[a]["accuracy"])
pt_export = os.path.join(MODEL_DIR, "model_T08.pt")
torch.save(pt_models[best_pt_arch], pt_export)
print(f"Saved best PyTorch model ({best_pt_arch}) to {pt_export}")

## TensorFlow / Keras

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

print("TensorFlow:", tf.__version__)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset="training",
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset="validation",
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

tf_class_names = train_ds.class_names
assert tf_class_names == class_names, (tf_class_names, class_names)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])
rescale = layers.Rescaling(1.0 / 255.0)

train_ds = train_ds.map(lambda x, y: (augment(rescale(x)), y), num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (rescale(x), y), num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (rescale(x), y), num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
TF_BASE_LAYER = {
    "resnet50": "resnet50",
    "efficientnet_b0": "efficientnetb0",
}


def _tf_gradcam_layer_name(base_model, architecture: str) -> str:
    """Pick a spatial (4D) layer for Grad-CAM (Keras 3-safe)."""
    preferred = {
        "resnet50": ["conv5_block3_out"],
        "efficientnet_b0": ["top_activation", "block7a_expand_activation"],
    }.get(architecture, [])
    for name in preferred:
        try:
            base_model.get_layer(name)
            return name
        except ValueError:
            continue
    for layer in reversed(base_model.layers):
        out_shape = getattr(layer, "output_shape", None)
        if out_shape is None and getattr(layer, "output", None) is not None:
            out_shape = layer.output.shape
        if out_shape is not None and len(out_shape) == 4:
            return layer.name
    raise ValueError(f"No Grad-CAM layer found for {architecture}")


def build_tf_model(architecture: str, num_classes: int):
    architecture = architecture.lower()
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

    if architecture == "resnet50":
        base = ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)
    elif architecture == "efficientnet_b0":
        base = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inputs)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    last_conv_name = _tf_gradcam_layer_name(base, architecture)

    base.trainable = False
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs=inputs, outputs=outputs, name=f"{architecture}_classifier")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base, last_conv_name


def load_tf_classifier(architecture: str):
    architecture = architecture.lower()
    model = tf.keras.models.load_model(tf_checkpoint_path(architecture))
    base = model.get_layer(TF_BASE_LAYER[architecture])
    meta_path = tf_meta_path(architecture)
    if os.path.isfile(meta_path):
        conv_name = _load_json(meta_path)["gradcam_layer"]
    else:
        conv_name = _tf_gradcam_layer_name(base, architecture)
    return model, base, conv_name


def evaluate_tf_model(model, test_ds):
    y_true, y_pred = [], []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    report = classification_report(y_true, y_pred, target_names=class_names)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "report": report,
        "y_true": y_true,
        "y_pred": y_pred,
    }


In [ ]:
tf_models = {}
tf_bases = {}
tf_conv_layers = {}
tf_histories = {}
tf_metrics = {}

for arch in ARCHITECTURES:
    print("\n" + "=" * 60)
    ckpt = tf_checkpoint_path(arch)
    hist_path = tf_history_path(arch)
    meta_path = tf_meta_path(arch)

    legacy_keras = os.path.join(MODEL_DIR, "model_T08.keras")
    if (
        not RETRAIN
        and not os.path.isfile(ckpt)
        and arch == "resnet50"
        and os.path.isfile(legacy_keras)
    ):
        print(f"Adopting legacy TensorFlow export for {arch}: {legacy_keras}")
        model = tf.keras.models.load_model(legacy_keras)
        base = model.get_layer(TF_BASE_LAYER[arch])
        conv_name = _tf_gradcam_layer_name(base, arch)
        model.save(ckpt)
        _save_json(meta_path, {"gradcam_layer": conv_name})
        print(f"Wrote per-arch checkpoint: {ckpt}")

    if not RETRAIN and os.path.isfile(ckpt):
        print(f"TensorFlow — loading cached {arch}")
        print("=" * 60)
        model, base, conv_name = load_tf_classifier(arch)
        if os.path.isfile(hist_path):
            tf_histories[arch] = _load_json(hist_path)
        else:
            tf_histories[arch] = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
            print("(No history JSON — training curves empty for this arch.)")
    else:
        print(f"TensorFlow — training {arch}")
        print("=" * 60)
        model, base, conv_name = build_tf_model(arch, len(class_names))
        history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        tf_histories[arch] = history.history
        model.save(ckpt)
        _save_json(hist_path, tf_histories[arch])
        _save_json(meta_path, {"gradcam_layer": conv_name})
        print(f"Saved checkpoint: {ckpt}")

    tf_models[arch] = model
    tf_bases[arch] = base
    tf_conv_layers[arch] = conv_name
    tf_metrics[arch] = evaluate_tf_model(model, test_ds)

    print(f"\nTest accuracy ({arch}): {tf_metrics[arch]['accuracy']:.4f}")
    print(tf_metrics[arch]["report"])
    print("Grad-CAM layer:", conv_name)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, arch in enumerate(ARCHITECTURES):
    h = tf_histories[arch]
    axes[i, 0].plot(h["loss"], label="Train")
    axes[i, 0].plot(h["val_loss"], label="Val")
    axes[i, 0].set_title(f"{arch} — loss")
    axes[i, 0].legend()

    axes[i, 1].plot(h["accuracy"], label="Train")
    axes[i, 1].plot(h["val_accuracy"], label="Val")
    axes[i, 1].set_title(f"{arch} — accuracy")
    axes[i, 1].legend()

plt.tight_layout()
plt.show()

tf_summary = [
    {
        "framework": "tensorflow",
        "architecture": arch,
        "accuracy": tf_metrics[arch]["accuracy"],
        "precision": tf_metrics[arch]["precision"],
        "recall": tf_metrics[arch]["recall"],
        "f1": tf_metrics[arch]["f1"],
    }
    for arch in ARCHITECTURES
]
print("TensorFlow comparison:")
for row in tf_summary:
    print(row)

In [ ]:
import cv2


def make_tf_gradcam_heatmap(img_array, model, base_model, last_conv_layer_name, pred_index=None):
    """img_array: float32 batch in [0, 1], shape (1, H, W, 3)."""
    last_conv_layer = base_model.get_layer(last_conv_layer_name)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[last_conv_layer.output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    h, w = img_array.shape[1], img_array.shape[2]
    heatmap = cv2.resize(heatmap, (w, h))
    return heatmap, int(pred_index)


def overlay_heatmap(rgb_img, heatmap, alpha=0.45):
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    blended = (1 - alpha) * (rgb_img * 255) + alpha * heatmap_color
    return np.clip(blended, 0, 255).astype(np.uint8)


sample_images, sample_labels = next(iter(test_ds.take(1)))
sample_images = sample_images.numpy()
sample_labels = sample_labels.numpy()

fig, axes = plt.subplots(len(ARCHITECTURES), 2, figsize=(10, 4 * len(ARCHITECTURES)))
if len(ARCHITECTURES) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, arch in enumerate(ARCHITECTURES):
    model = tf_models[arch]
    base = tf_bases[arch]
    conv_name = tf_conv_layers[arch]

    img = sample_images[0:1].astype(np.float32)
    rgb = np.clip(img[0], 0.0, 1.0)
    true_label = int(sample_labels[0])

    heatmap, pred_class = make_tf_gradcam_heatmap(img, model, base, conv_name)
    overlay = overlay_heatmap(rgb, heatmap)

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"{arch} — original ({class_names[true_label]})")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title(f"Grad-CAM → {class_names[pred_class]}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cross_framework = pt_summary + tf_summary
print("\nCross-framework comparison (same architectures):")
for row in cross_framework:
    print(
        f"{row['framework']:12} {row['architecture']:18} "
        f"acc={row['accuracy']:.4f} f1={row['f1']:.4f}"
    )

best_tf_arch = max(ARCHITECTURES, key=lambda a: tf_metrics[a]["accuracy"])
tf_export = os.path.join(MODEL_DIR, "model_T08.keras")
tf_models[best_tf_arch].save(tf_export)
print(f"Saved best TensorFlow model ({best_tf_arch}) to {tf_export}")
